In [2]:
import pandas as pd

In [3]:
df1 = pd.read_excel("data/2021.xlsx").drop(['Round', 'Quota'], axis=1)
df2 = pd.read_excel("data/2022.xlsx").drop(['Round', 'Quota'], axis=1)
df3 = pd.read_excel("data/2023.xlsx").drop(['Round', 'Quota'], axis=1)
df4 = pd.read_excel("data/2024.xlsx").drop(['Round', 'Quota'], axis=1)
df5 = pd.read_excel("data/2025.xlsx").drop(['Quota'], axis=1)

df1.dropna(inplace=True)
df2.dropna(inplace=True)
df3.dropna(inplace=True)
df4.dropna(inplace=True)
df5.dropna(inplace=True)

In [4]:
df1

,Year,Institute,Academic Program Name,Seat Type,Gender,Opening Rank,Closing Rank
0,2021,Indian Institute of Technology Bhubaneswar,"Civil Engineering (4 Years, Bachelor of Techno...",OPEN,Gender-Neutral,8471,12396
1,2021,Indian Institute of Technology Bhubaneswar,"Civil Engineering (4 Years, Bachelor of Techno...",OPEN,Female-only,16998,21029
2,2021,Indian Institute of Technology Bhubaneswar,"Civil Engineering (4 Years, Bachelor of Techno...",EWS,Gender-Neutral,1603,1760
3,2021,Indian Institute of Technology Bhubaneswar,"Civil Engineering (4 Years, Bachelor of Techno...",EWS,Female-only,3536,3536
4,2021,Indian Institute of Technology Bhubaneswar,"Civil Engineering (4 Years, Bachelor of Techno...",OBC-NCL,Gender-Neutral,3044,4260
...,...,...,...,...,...,...,...
9173,2021,"North-Eastern Hill University, Shillong","Information Technology (4 Years, Bachelor of T...",OPEN,Gender-Neutral,59431,68295
9174,2021,"North-Eastern Hill University, Shillong","Information Technology (4 Years, Bachelor of T...",EWS,Gender-Neutral,10427,11089
9175,2021,"North-Eastern Hill University, Shillong","Information Technology (4 Years, Bachelor of T...",OBC-NCL,Gender-Neutral,21224,21577
9176,2021,"North-Eastern Hill University, Shillong","Information Technology (4 Years, Bachelor of T...",SC,Gender-Neutral,8907,11617


In [5]:
df5['Year'] = 2025
df5

,Institute,Academic Program Name,Seat Type,Gender,Opening Rank,Closing Rank,Year
0,Indian Institute of Technology Bhubaneswar,"Civil Engineering (4 Years, Bachelor of Techno...",OPEN,Gender-Neutral,10922,16156,2025
1,Indian Institute of Technology Bhubaneswar,"Civil Engineering (4 Years, Bachelor of Techno...",OPEN,Female-only (including Supernumerary),19820,23960,2025
2,Indian Institute of Technology Bhubaneswar,"Civil Engineering (4 Years, Bachelor of Techno...",EWS,Gender-Neutral,2214,2346,2025
3,Indian Institute of Technology Bhubaneswar,"Civil Engineering (4 Years, Bachelor of Techno...",EWS,Female-only (including Supernumerary),3688,3698,2025
4,Indian Institute of Technology Bhubaneswar,"Civil Engineering (4 Years, Bachelor of Techno...",OBC-NCL,Gender-Neutral,4402,5414,2025
...,...,...,...,...,...,...,...
11939,Shri G. S. Institute of Technology and Science...,Electronics and Instrumentation Engineering (4...,OPEN,Gender-Neutral,48831,51031,2025
11940,Shri G. S. Institute of Technology and Science...,Electronics and Telecommunication Engineering ...,OPEN,Gender-Neutral,35170,41294,2025
11941,Shri G. S. Institute of Technology and Science...,Industrial and Production Engineering (4 Years...,OPEN,Gender-Neutral,67155,71193,2025
11942,Shri G. S. Institute of Technology and Science...,"Information Technology (4 Years, Bachelor of T...",OPEN,Gender-Neutral,28578,32619,2025


In [6]:
def prepare_rag_data(df):
    rules = {
        'Indian Institute of Technology': 'IIT',
        'National Institute of Technology': 'NIT',
        'Indian Institutes of Information Technology': 'IIIT'
    }
    
    for keyword, abbr in rules.items():
        # Mask ensures we don't process rows that already contain the abbreviation (e.g., "IIT")
        mask = df['Institute'].str.contains(keyword, na=False) & ~df['Institute'].str.contains(abbr, na=False)
        
        # Regex explanation:
        # (keyword) captures the base name (Group 1)
        # (.*) captures everything after it, e.g., " Madras" (Group 2)
        # \g<0> represents the entire original match, and we append (abbr + Group 2) to it.
        df.loc[mask, 'Institute'] = df.loc[mask, 'Institute'].str.replace(
            rf'({keyword})(.*)', 
            rf'\g<0> ({abbr}\2)', 
            regex=True
        )
        
    # Vectorized creation of highly contextual statements for RAG
    df['RAG_Context'] = (
        "In " + df['Year'].astype(str) + ", admission to the " + df['Academic Program Name'] + 
        " program at " + df['Institute'] + " under the " + df['Seat Type'] + " category for " + 
        df['Gender'] + " candidates had an opening rank of " + df['Opening Rank'].astype(str) + 
        " and a closing rank of " + df['Closing Rank'].astype(str) + "."
    )
    return df

# Apply the transformation to all 5 dataframes efficiently
dfs = [df1, df2, df3, df4, df5]
for df in dfs:
    prepare_rag_data(df)

In [7]:
df1['Institute'].unique()

array(['Indian Institute of Technology Bhubaneswar (IIT Bhubaneswar)',
       'Indian Institute of Technology Bombay (IIT Bombay)',
       'Indian Institute of Technology Mandi (IIT Mandi)',
       'Indian Institute of Technology Delhi (IIT Delhi)',
       'Indian Institute of Technology Indore (IIT Indore)',
       'Indian Institute of Technology Kharagpur (IIT Kharagpur)',
       'Indian Institute of Technology Hyderabad (IIT Hyderabad)',
       'Indian Institute of Technology Jodhpur (IIT Jodhpur)',
       'Indian Institute of Technology Kanpur (IIT Kanpur)',
       'Indian Institute of Technology Madras (IIT Madras)',
       'Indian Institute of Technology Gandhinagar (IIT Gandhinagar)',
       'Indian Institute of Technology Patna (IIT Patna)',
       'Indian Institute of Technology Roorkee (IIT Roorkee)',
       'Indian Institute of Technology (ISM) Dhanbad (IIT (ISM) Dhanbad)',
       'Indian Institute of Technology Ropar (IIT Ropar)',
       'Indian Institute of Technology (BHU

In [8]:
# List of target institutes as they appear after your previous transformation
target_institutes = [
    'Indian Institute of Technology Madras (IIT Madras)', 
    'Indian Institute of Technology Bombay (IIT Bombay)', 
    'Indian Institute of Technology Delhi (IIT Delhi)', 
    'Indian Institute of Technology Kharagpur (IIT Kharagpur)', 
    'Indian Institute of Technology Kanpur (IIT Kanpur)', 
    'Indian Institute of Technology Roorkee (IIT Roorkee)', 
    'Indian Institute of Technology Guwahati (IIT Guwahati)'
]

# 1. Filter each dataframe and store them in a list
filtered_dfs = [df[df['Institute'].isin(target_institutes)] for df in [df1, df2, df3, df4, df5]]

# 2. Merge them top-to-bottom into a single dataframe
df_merged = pd.concat(filtered_dfs, ignore_index=True)

# Generate counts for each institute in the merged dataframe
institute_counts = df_merged['Institute'].value_counts()

# Display the counts to verify
print("Counts for each Institute in df_merged:")
print(institute_counts)
    
# Optional: verify the result
print(f"Total rows in merged dataframe: {len(df_merged)}")
print(df_merged['Institute'].unique())

# Check for any missing institutes from your target list
missing = set(target_institutes) - set(df_merged['Institute'].unique())
if not missing:
    print("\nVerification Successful: All target institutes are included.")
else:
    print(f"\nWarning: The following institutes are missing from the dataframe: {missing}")

Counts for each Institute in df_merged:
Institute
Indian Institute of Technology Kharagpur (IIT Kharagpur)    1626
Indian Institute of Technology Delhi (IIT Delhi)            1027
Indian Institute of Technology Roorkee (IIT Roorkee)         966
Indian Institute of Technology Bombay (IIT Bombay)           913
Indian Institute of Technology Madras (IIT Madras)           809
Indian Institute of Technology Kanpur (IIT Kanpur)           794
Indian Institute of Technology Guwahati (IIT Guwahati)       647
Name: count, dtype: int64
Total rows in merged dataframe: 6782
['Indian Institute of Technology Bombay (IIT Bombay)'
 'Indian Institute of Technology Delhi (IIT Delhi)'
 'Indian Institute of Technology Kharagpur (IIT Kharagpur)'
 'Indian Institute of Technology Kanpur (IIT Kanpur)'
 'Indian Institute of Technology Madras (IIT Madras)'
 'Indian Institute of Technology Roorkee (IIT Roorkee)'
 'Indian Institute of Technology Guwahati (IIT Guwahati)']

Verification Successful: All target instit

In [9]:
df_merged

,Year,Institute,Academic Program Name,Seat Type,Gender,Opening Rank,Closing Rank,RAG_Context
0,2021,Indian Institute of Technology Bombay (IIT Bom...,"Aerospace Engineering (4 Years, Bachelor of Te...",OPEN,Gender-Neutral,123,2118,"In 2021, admission to the Aerospace Engineerin..."
1,2021,Indian Institute of Technology Bombay (IIT Bom...,"Aerospace Engineering (4 Years, Bachelor of Te...",OPEN,Female-only,702,4419,"In 2021, admission to the Aerospace Engineerin..."
2,2021,Indian Institute of Technology Bombay (IIT Bom...,"Aerospace Engineering (4 Years, Bachelor of Te...",OPEN (PwD),Gender-Neutral,7,7,"In 2021, admission to the Aerospace Engineerin..."
3,2021,Indian Institute of Technology Bombay (IIT Bom...,"Aerospace Engineering (4 Years, Bachelor of Te...",EWS,Gender-Neutral,342,476,"In 2021, admission to the Aerospace Engineerin..."
4,2021,Indian Institute of Technology Bombay (IIT Bom...,"Aerospace Engineering (4 Years, Bachelor of Te...",EWS,Female-only,931,1501,"In 2021, admission to the Aerospace Engineerin..."
...,...,...,...,...,...,...,...,...
6777,2025,Indian Institute of Technology Guwahati (IIT G...,"Mechanical Engineering (4 Years, Bachelor of T...",OBC-NCL,Female-only (including Supernumerary),5215,5800,"In 2025, admission to the Mechanical Engineeri..."
6778,2025,Indian Institute of Technology Guwahati (IIT G...,"Mechanical Engineering (4 Years, Bachelor of T...",SC,Gender-Neutral,664,1240,"In 2025, admission to the Mechanical Engineeri..."
6779,2025,Indian Institute of Technology Guwahati (IIT G...,"Mechanical Engineering (4 Years, Bachelor of T...",SC,Female-only (including Supernumerary),1841,2519,"In 2025, admission to the Mechanical Engineeri..."
6780,2025,Indian Institute of Technology Guwahati (IIT G...,"Mechanical Engineering (4 Years, Bachelor of T...",ST,Gender-Neutral,440,610,"In 2025, admission to the Mechanical Engineeri..."


In [ ]:
from sentence_transformers import SentenceTransformer
import time

# 1. Load the model (this downloads the model locally on the first run)
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")

def generate_embeddings_with_retry(texts, max_retries=3):
    """Generates embeddings with a retry mechanism for system robustness."""
    for attempt in range(max_retries):
        try:
            # batch_size handles large data efficiently; show_progress_bar gives you visual feedback
            return model.encode(texts, batch_size=64, show_progress_bar=True)
        except Exception as e:
            print(f"Attempt {attempt + 1} failed with error: {e}")
            time.sleep(2)  # Brief pause before retrying
            
    raise RuntimeError("Embedding generation failed after multiple attempts.")

# 2. Extract texts and generate embeddings 
texts_to_encode = df_merged['RAG_Context'].tolist()
embeddings = generate_embeddings_with_retry(texts_to_encode)

# 3. Save to the new column
# Convert the numpy array of embeddings into a list of arrays so it fits in a pandas column
df_merged['Embeddings'] = list(embeddings)

# 4. Verification Step
missing_embeddings = df_merged['Embeddings'].isna().sum()

# Check if there are no missing values AND that the first embedding actually has data
if missing_embeddings == 0 and len(df_merged.loc[0, 'Embeddings']) > 0:
    print(f"✅ Verification Successful: All {len(df_merged)} rows have successfully generated embeddings.")
    df_merged.to_csv("data/data_with_embeddings.csv", index=False)
else:
    print(f"❌ Warning: There are {missing_embeddings} rows missing embeddings.")

c:\Users\niksh\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\niksh\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\niksh\.cache\huggingface\hub\models--Qwen--Qwen3-Embedding-0.6B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as a

KeyboardInterrupt: 